# 02 · Forward pass and decoding

Run a looming stimulus through a network and read out whether it sees a hit,
and where the object is and how fast it moves.

Needs `flyvis` and `pytorch3d`, plus a RAFT checkout for the RAFT ids.

In [ ]:
import torch, numpy as np, matplotlib.pyplot as plt
from flynet_robustness.models import get_flynet, get_stimulus_and_return_activity
from flynet_robustness.decoders import load_decoder, load_decoder_pos_vel
from flynet_robustness.textures import get_texture

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model1, model2, idxs1, idxs2 = get_flynet('FlyNet', device=device)   # or 'CTRNN', 'sCTRNN'
print('decoded T4/T5 units:', len(idxs1))

In [ ]:
activity, speed, angle, position = get_stimulus_and_return_activity(
    model1, idxs1, model2, idxs2, hit=True, shape='Sphere',
    texture=get_texture('Black'), time=1.0, device=device)
print('activity', tuple(activity.shape))

## Hit or miss

In [ ]:
from flynet_robustness.plotting import plot_p_hit_trace
decoder = load_decoder('FlyNet', device=device)
p_hit = torch.sigmoid(decoder(activity.flatten(end_dim=1).to(device))).detach().cpu().numpy()
plot_p_hit_trace(p_hit, color='forestgreen');

## Position and velocity

These regressors live in `decoders/FlyNet/`.

In [ ]:
vel = load_decoder_pos_vel('FlyNet', load_velocity=True, device=device)
pred = vel(activity.flatten(end_dim=1).to(device)).detach().cpu().numpy()
print('predicted velocity', pred.shape)

## The same for any network

`load_network` gives the `(net, decoder)` pair for any id, and
`network_activity` renders a stimulus and returns what that network feeds its
decoder: 5,768 T4/T5 units for FlyNet, CTRNN and sCTRNN, 7,688 for the CNNs,
6,272 for RAFT. The CNNs and RAFT read a sliding window of frames, so they
return fewer samples than there are frames.

In [ ]:
import json
from flynet_robustness import config
from flynet_robustness.models import load_network
from flynet_robustness.dataset import network_activity

t = json.load(open('data/configs/attack_trajectories.json'))[0]   # one every network detects

for network_id in config.PAPER_NETWORKS:
    net, dec = load_network(network_id, device=device)
    act, pos = network_activity(net, network_id, hit=True, shape='Sphere',
                                texture=get_texture('Black'), time=1.0,
                                stop_margin_hit=0.6, **t, device=device)
    p = torch.sigmoid(dec(act.to(device))).detach().cpu().numpy().ravel()
    q = len(p) // 4
    print(f'{network_id:16s} activity {str(tuple(act.shape)):14s} '
          f'p(hit) first quarter {p[:q].mean():.2f} -> last {p[-q:].mean():.2f}')
    del net, dec